# POL SAOCOM demo

In [ ]:
"""
File Name: pol.ipynb

Authors:
    Juan M. Lopez-Sanchez, Alejandro Mestre-Quereda  
    University of Alicante

Date: June 2025

Description:
------------
This script performs the generation of POL products at a single date 
from dual-pol SAOCOM images in CSLC format.
Options and configuration parameters about some processing steps can be set
by the user.

Dependencies:
-------------
- os
- numpy
- rosel

Usage:
------
To use this script, define the following parameters:
    
    - common_input_folder: Path to the main input folder with preprocessed CLSC data
    - common_output_folder: Path to the main output folder in which the POL products will be saved
    - site: 'Naples'', 'Salamanca', ''BlackForest' or 'Brest'
    - orbit: 'ASC' or 'DSC'
    - beam: 'S3' or 'S4'
    - estimation_type = Select one option: 'MULTI-LOOK', 'BOXCAR' or 'SHIROMA'
    - kernel_size = [ 9, 5 ] # [azimuth,range] Required for 'MULTI-LOOK' and 'BOXCAR'
    - acquisition_date: One of the timestamps listed in the cslc['acquisition_date'] variable, e.g. '2024-08-23T06:10:49.955322'

"""

In [ ]:
import os
import numpy as np

import rosel.io
import rosel.computation
import rosel.shiroma

######################################
# PARAMETERS
######################################

common_input_folder = '/media/s3-ard-rose-l/SAOCOM'
common_output_folder = '/media/s3-ard-rose-l/PHASE_1/DEMONSTRATOR/POL'

# Site, orbit pass, and beam

site = 'Salamanca' # Naples, Salamanca, BlackForest, Brest
orbit = 'ASC'   # ASC, DSC
beam = 'S4'      # S3, S4

# Processing options

estimation_type = 'BOXCAR' # 'MULTI-LOOK' or 'BOXCAR' or 'SHIROMA'

kernel_size = [ 9, 5 ] # [azimuth,range] Required for 'MULTI-LOOK' and 'BOXCAR'

#%%###################################
# MAIN PROGRAM
######################################

if __name__ == "__main__":

    # Input folder with pre-processsed images in CSLC format

    input_folder = os.path.join( common_input_folder, site, orbit, beam )
    in_path = rosel.io.normpath( input_folder )

    # Output folder
    
    common_out_path = os.path.join( common_output_folder, site, orbit, beam, estimation_type )

    if estimation_type != 'SHIROMA':
        common_out_path = os.path.join( common_out_path, str(kernel_size[0]) + 'x' + str(kernel_size[1]) )

In [ ]:
    
    subfolder_list = [ os.path.basename(f.path) for f in os.scandir(input_folder) if f.is_dir() ]
    
    for subfolder in subfolder_list:
        
        if subfolder.endswith('_VV'):
            cslc_path_vv = os.path.join( in_path, subfolder, 'coreg')
    
        if subfolder.endswith('_VH'):
            cslc_path_vh = os.path.join( in_path, subfolder, 'coreg')
            
        if subfolder == 'SHIROMA':
            shiroma_path = os.path.join( in_path, 'SHIROMA')
    
    selection_path = os.path.join( shiroma_path, 'selection' )
    global_values_path = os.path.join( shiroma_path, 'global_values' )
    rtc_ap_path = os.path.join( shiroma_path, 'rtc_ap' )
    
    ########################
    # PROCESSING
    ########################

In [ ]:
    # Check the available acquistions to select one:

    cslc = rosel.io.read(cslc_path_vv, consolidate_metadata = False)
    cslc['acquisition_date']
    

In [ ]:
    # The user needs to select one of the dates from the list.
    # For instance:

    acquisition_date = '2024-09-24T06:10:51.209735' # ASC/S4 Salamanca
    
    date_txt = acquisition_date[:10] # Only the string with the date
    
    print('Date: ', date_txt)
    
    out_path = os.path.join( common_out_path, date_txt )

In [ ]:
    
    print('Read SLC images...')
    
    slc_vv = rosel.io.read(cslc_path_vv, consolidate_metadata = False).cstack.sel(acquisition_date=acquisition_date)
    slc_vh = rosel.io.read(cslc_path_vh, consolidate_metadata = False).cstack.sel(acquisition_date=acquisition_date)
    
    # Read auxiliary layers for flattening and geocoding
    
    selection = rosel.io.read( selection_path, consolidate_metadata = False )
    A = rosel.io.read( rtc_ap_path, mask_and_scale = False, consolidate_metadata = False ).v

In [ ]:
    
    print('Flatten the SLC values of both channels ...')
    
    slc_vv_flat = rosel.shiroma.flatten_radar_image(slc_vv, A, data_path = f"{out_path}/flattened_slc_vv", read=True)
    
    slc_vh_flat = rosel.shiroma.flatten_radar_image(slc_vh, A, data_path = f"{out_path}/flattened_slc_vh", read=True)

In [ ]:
    
    print('Covariance matrix formation: cross-products and estimation ...')
    
    if estimation_type == 'MULTI-LOOK': 

        multilook = {"azimuth_time": kernel_size[0], "slant_range_time": kernel_size[1]}
    
        C11 = (np.abs((slc_vv_flat * slc_vv_flat.conj())).astype(np.float32).coarsen(azimuth_time=multilook["azimuth_time"], slant_range_time=multilook["slant_range_time"], boundary="pad")
            .mean().chunk(azimuth_time=300, slant_range_time=300))
        
        C22 = (np.abs((slc_vh_flat * slc_vh_flat.conj())).astype(np.float32).coarsen(azimuth_time=multilook["azimuth_time"], slant_range_time=multilook["slant_range_time"], boundary="pad")
            .mean().chunk(azimuth_time=300, slant_range_time=300))
        
        C12 = (slc_vv_flat * slc_vh_flat.conj()).coarsen(azimuth_time=multilook["azimuth_time"], slant_range_time=multilook["slant_range_time"], boundary="pad").mean().chunk(azimuth_time=300, slant_range_time=300)
    
    elif estimation_type == 'BOXCAR':

        boxcar_kernel = {"azimuth_time": kernel_size[0], "slant_range_time": kernel_size[1]}

        data = np.abs((slc_vv_flat * slc_vv_flat.conj())).astype(np.float32)
        C11  = data.rolling( azimuth_time=boxcar_kernel["azimuth_time"], slant_range_time=boxcar_kernel["slant_range_time"], center=True).mean().chunk({"azimuth_time": 1024, "slant_range_time": 1024})

        data = np.abs((slc_vh_flat * slc_vh_flat.conj())).astype(np.float32)
        C22 = data.rolling( azimuth_time=boxcar_kernel["azimuth_time"], slant_range_time=boxcar_kernel["slant_range_time"], center=True).mean().chunk({"azimuth_time": 1024, "slant_range_time": 1024})   
               
        data = slc_vv_flat * slc_vh_flat.conj()
        C12 = data.rolling( azimuth_time=boxcar_kernel["azimuth_time"], slant_range_time=boxcar_kernel["slant_range_time"], center=True).mean().chunk({"azimuth_time": 1024, "slant_range_time": 1024})         
        
    elif estimation_type == 'SHIROMA':

        C11 = np.abs((slc_vv_flat * slc_vv_flat.conj())).astype(np.float32)

        C22 = np.abs((slc_vh_flat * slc_vh_flat.conj())).astype(np.float32)
               
        C12 = slc_vv_flat * slc_vh_flat.conj()
        
    else:
        
        print('Wrong input for type of estimation of the COV matrix.')
        print("There are 3 options: 'MULTI-LOOK', 'BOXCAR' or 'SHIROMA'")

In [ ]:
    print('Save COV entries ...')
    
    rosel.io.write(C11.to_dataset(name="v"), f"{out_path}/COV/C11")
    rosel.io.write(C22.to_dataset(name="v"), f"{out_path}/COV/C22")
    rosel.io.write(C12.to_dataset(name="v"), f"{out_path}/COV/C12")

In [ ]:
    
    #   Geocoding is carried out using Shiroma's approach
    #   In case of 'MULTI-LOOK' estimation, the output grid could be coarsened
    #   Otherwise, the output grid posting is similar to SLC resolution
    
    print('Geocoding ...')
    
    if estimation_type == 'MULTI-LOOK': 
    
        gC11 = rosel.shiroma.map_geo_ap(
            c_path=f"{out_path}/COV/C11",
            simulation=selection,
            data_path_shiroma=shiroma_path,
            data_path_geo_ap=f"{out_path}/GCOV/C11",
            multilook=multilook,
            read=True,
            enable_dask=True,
        )
            
        gC22 = rosel.shiroma.map_geo_ap(
            c_path=f"{out_path}/COV/C22",
            simulation=selection,
            data_path_shiroma=shiroma_path,
            data_path_geo_ap=f"{out_path}/GCOV/C22",
            multilook=multilook,
            read=True,
            enable_dask=True,
        )
            
        gC12 = rosel.shiroma.map_geo_ap(
            c_path=f"{out_path}/COV/C12",
            simulation=selection,
            data_path_shiroma=shiroma_path,
            data_path_geo_ap=f"{out_path}/GCOV/C12",
            multilook=multilook,
            read=True,
            enable_dask=True,
        )
    
    else:
        
        gC11 = rosel.shiroma.map_geo_ap(
            c_path=f"{out_path}/COV/C11",
            simulation=selection,
            data_path_shiroma=shiroma_path,
            data_path_geo_ap=f"{out_path}/GCOV/C11",
            read=True,
            enable_dask=True,
        )

        gC22 = rosel.shiroma.map_geo_ap(
            c_path=f"{out_path}/COV/C22",
            simulation=selection,
            data_path_shiroma=shiroma_path,
            data_path_geo_ap=f"{out_path}/GCOV/C22",
            read=True,
            enable_dask=True,
        )

        gC12 = rosel.shiroma.map_geo_ap(
            c_path=f"{out_path}/COV/C12",
            simulation=selection,
            data_path_shiroma=shiroma_path,
            data_path_geo_ap=f"{out_path}/GCOV/C12",
            read=True,
            enable_dask=True,
        )

